# XLZD Shell Theta CNP Workflow: Minibatch

This notebook runs the shell-theta minibatch CNP experiment on the prepared shell-theta HDF5 files.

It uses:

- `xlzd_shell_theta/settings_shell_minibatch.yaml`
- `xlzd_shell_theta/settings_shell_validation_minibatch.yaml`

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, display


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "prepare_resum_data.py").exists() and (candidate / "README.md").exists():
            return candidate
    raise RuntimeError("Could not find the XLZD repo root from the current working directory.")


REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)

if str(REPO_ROOT / "src" / "run_cnp") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src" / "run_cnp"))

from cnp_clean_pipeline import load_runtime_config, predict_cnp, train_cnp

EXPERIMENT = "shell_theta_minibatch"
CONFIG_PATH = REPO_ROOT / "xlzd_shell_theta" / "settings_shell_minibatch.yaml"
VALIDATION_CONFIG_PATH = REPO_ROOT / "xlzd_shell_theta" / "settings_shell_validation_minibatch.yaml"

SEED = 42
STEPS_PER_EPOCH = 5000
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.0
REPR_DIM = 32
HIDDEN = 128
DROPOUT = 0.1
MONITOR_EVERY = 5000
MC_SAMPLES = 30
CHUNK_SIZE = 20000

print(f"Repo root: {REPO_ROOT}")
print(f"Experiment: {EXPERIMENT}")
print(f"Training config: {CONFIG_PATH}")
print(f"Validation config: {VALIDATION_CONFIG_PATH}")


## 1. Load And Inspect The Runtime Config

This cell loads the shell-theta CNP settings and shows the folders the notebook will use.

In [ ]:
runtime = load_runtime_config(CONFIG_PATH, seed=SEED)
validation_runtime = load_runtime_config(VALIDATION_CONFIG_PATH, seed=SEED)

summary = pd.DataFrame(
    {
        "field": [
            "version",
            "train_dir",
            "predict_dirs",
            "theta_headers",
            "phi_headers",
            "target_headers",
            "training_mode",
            "epochs",
            "steps_per_epoch",
            "context_ratio",
            "out_dir",
        ],
        "value": [
            runtime.version,
            str(runtime.train_dir),
            ", ".join(str(p) for p in runtime.predict_dirs),
            ", ".join(runtime.theta_headers),
            ", ".join(runtime.phi_headers),
            ", ".join(runtime.target_headers),
            runtime.training_mode,
            runtime.epochs,
            runtime.steps_per_epoch,
            runtime.context_ratio,
            str(runtime.out_dir),
        ],
    }
)
summary


## 2. Train The CNP

This trains the CNP on the LF training shell-theta H5 files and writes the model, history CSV, and plots to `data/out/cnp`.

In [ ]:
train_result = train_cnp(
    runtime,
    steps_per_epoch=STEPS_PER_EPOCH,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    repr_dim=REPR_DIM,
    hidden=HIDDEN,
    dropout=DROPOUT,
    monitor_every=MONITOR_EVERY,
    show_monitor_plots=True,
)

pd.DataFrame(
    {
        "artifact": ["model_path", "history_csv", "history_plot", "sample_plot"],
        "path": [
            str(train_result.model_path),
            str(train_result.history_csv),
            str(train_result.history_plot),
            str(train_result.sample_plot),
        ],
    }
)


## 3. Predict On Training LF + HF

In [ ]:
predict_result_train = predict_cnp(
    runtime,
    model_path=train_result.model_path,
    mc_samples=MC_SAMPLES,
    chunk_size=CHUNK_SIZE,
)

train_pred_df = pd.read_csv(predict_result_train.csv_path)
display(train_pred_df.head())
display(train_pred_df.describe(include="all"))


In [ ]:
display(Image(filename=str(predict_result_train.heatmap_path)))
display(Image(filename=str(predict_result_train.error_heatmap_path)))


## 4. Predict On Held-Out HF Validation Files

In [ ]:
predict_result_validation = predict_cnp(
    validation_runtime,
    model_path=train_result.model_path,
    mc_samples=MC_SAMPLES,
    chunk_size=CHUNK_SIZE,
)

validation_pred_df = pd.read_csv(predict_result_validation.csv_path)
display(validation_pred_df.head())
display(validation_pred_df.describe(include="all"))


In [ ]:
display(Image(filename=str(predict_result_validation.heatmap_path)))
display(Image(filename=str(predict_result_validation.error_heatmap_path)))


## 5. Key Output Files

The notebook writes the same artifact types as the standard CNP workflow, but for the shell-theta version.